# EUW Periodograms of Hourly PCA Scores

This notebook asks whether the hourly PCA scores have regular cycles.
It computes Lomb-Scargle periodograms for PC1, PC2, and PC3 over a
6-48 hour period range.

## Setup

In Colab, the notebook mounts Google Drive and looks for the raw Riot
Parquet at the same shared-drive path used by the other release
notebooks: `/content/drive/Shareddrives/MSc_2026_Riot/db/riotData.parquet`.
The materialized `hourly_agg` cache is stored beside it as
`riot_local.duckdb`. The first run creates the cache; later Colab
sessions reuse it without rebuilding the aggregate table.

The notebook also needs the repository Python files. If they are not
already present in the runtime or Drive, the setup cell tries to clone
the student repository into `/content/MSC2026_LoL_Students`. Run the
single setup code cell below before any analysis cell.

In [ ]:
# Local users should normally use the uv environment from README.md.
# This cell only installs missing packages when the notebook is opened in Colab.
import importlib.util
import subprocess
import sys

MODULE_TO_PACKAGE = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "duckdb": "duckdb",
    "astropy": "astropy",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
    "joblib": "joblib",
}

missing = [
    package
    for module, package in MODULE_TO_PACKAGE.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("Notebook packages are available.")



from pathlib import Path
import importlib
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore[import-not-found]  # noqa: F401
    except ImportError:
        return False
    return True


IN_COLAB = running_in_colab()
if IN_COLAB:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")


# Override this if your repository folder has a different Colab/Drive location.
ROOT_OVERRIDE = None
REPO_URL = "https://github.com/wadelab/MSC2026_LoL_Students.git"
COLAB_REPO = Path(os.environ.get("LOL_REPO_PATH", "/content/MSC2026_LoL_Students"))


def find_repo_root() -> Path | None:
    if ROOT_OVERRIDE is not None:
        candidate = Path(ROOT_OVERRIDE).expanduser()
        if (candidate / "riot_analysis.py").exists():
            return candidate.resolve()
        raise FileNotFoundError(f"ROOT_OVERRIDE does not contain riot_analysis.py: {candidate}")

    candidates = list(Path.cwd().resolve().parents)
    candidates.insert(0, Path.cwd().resolve())
    candidates.extend(
        [
            COLAB_REPO,
            Path("/content/drive/MyDrive/MSC2026_LoL_Students"),
            Path("/content/drive/Shareddrives/MSc_2026_Riot/MSC2026_LoL_Students"),
        ]
    )
    for candidate in candidates:
        if (candidate / "riot_analysis.py").exists():
            return candidate.resolve()
    return None


ROOT = find_repo_root()
if IN_COLAB:
    if (COLAB_REPO / ".git").exists():
        print("Updating Colab repository clone...")
        subprocess.run(["git", "-C", str(COLAB_REPO), "pull", "--ff-only"], check=True)
    elif not COLAB_REPO.exists():
        print("Cloning repository files needed by the notebook...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO)], check=True)
    elif not (COLAB_REPO / "riot_analysis.py").exists():
        raise FileNotFoundError(
            f"{COLAB_REPO} exists but is not a usable repository clone. "
            "Remove or rename it, then rerun this setup cell."
        )
    if ROOT_OVERRIDE is None:
        ROOT = COLAB_REPO.resolve()
    else:
        ROOT = find_repo_root()

if ROOT is None or not (ROOT / "riot_analysis.py").exists():
    raise FileNotFoundError(
        "Could not find riot_analysis.py after repository setup. "
        "Set ROOT_OVERRIDE to a complete repository folder."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()

from riot_analysis import resolve_analysis_db_file

ANALYSIS_DB_FILE = resolve_analysis_db_file() if IN_COLAB else ROOT / "riot_local.duckdb"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 120

print(f"Repository root: {ROOT}")
print(f"riot_analysis.py: {ROOT / 'riot_analysis.py'}")
print(f"Running in Colab: {IN_COLAB}")
print(f"Persistent DuckDB cache: {ANALYSIS_DB_FILE}")

In [ ]:
from riot_analysis import (
    AnalysisConfig,
    GOOD_PCA_COLS,
    add_time_normalized_features,
    compute_pca,
    configure_plot_style,
    connect_analysis_database,
    filter_hourly_window,
    filter_metric_outliers,
    fixed_period_lag_table,
    load_hourly_metrics,
    lomb_scargle_summary,
    period_grid,
    style_axes,
)

configure_plot_style()

PLATFORM = "EUW1"
DB_FILE = ANALYSIS_DB_FILE
PARQUET_FILE = None

config = AnalysisConfig(
    platform=PLATFORM,
    max_hour_limit=10000,
    min_period_h=6,
    max_period_h=48,
    period_step_h=0.25,
    output_root=ROOT / "results",
)
conn = connect_analysis_database(DB_FILE, parquet_file=PARQUET_FILE)

In [ ]:
def hourly_pca_scores(conn, platform: str, config: AnalysisConfig) -> tuple[pd.DataFrame, dict]:
    hourly_metrics = load_hourly_metrics(conn, platform)
    hourly_metrics = filter_hourly_window(hourly_metrics, config.max_hour_limit)
    hourly_metrics, numeric_cols = add_time_normalized_features(hourly_metrics)
    hourly_metrics = filter_metric_outliers(hourly_metrics, numeric_cols)
    pca = compute_pca(hourly_metrics, numeric_cols, GOOD_PCA_COLS)

    scores = hourly_metrics.loc[pca["features"].index, ["hour_idx"]].reset_index(drop=True)
    scores["hours_since_start"] = scores["hour_idx"] - scores["hour_idx"].min()
    for i in range(3):
        scores[f"PC{i + 1}"] = pca["scores"][:, i]
    return scores, pca


pc_scores, pca = hourly_pca_scores(conn, PLATFORM, config)
display(pc_scores.head())

## Compute periodograms

Lomb-Scargle is useful here because it handles gaps in the hourly
sequence without requiring interpolation.

In [ ]:
frequency, period = period_grid(config)
periodogram_curves = {}
summary_rows = []
fixed_period_tables = []

for component in ["PC1", "PC2", "PC3"]:
    result = lomb_scargle_summary(
        pc_scores["hour_idx"].to_numpy(dtype=float),
        pc_scores[component].to_numpy(dtype=float),
        frequency,
        period,
    )
    periodogram_curves[component] = result
    summary_rows.append(
        {
            "component": component,
            "best_period_h": result["best_period"],
            "power_at_24h": result["power_24"],
        }
    )

    fit_table = fixed_period_lag_table(
        pc_scores["hours_since_start"].to_numpy(dtype=float),
        pc_scores[component].to_numpy(dtype=float),
        periods_h=(12.0, 24.0, 36.0, 48.0),
        robust=True,
    )
    fit_table.insert(0, "component", component)
    fixed_period_tables.append(fit_table)

periodogram_summary = pd.DataFrame(summary_rows)
fixed_period_summary = pd.concat(fixed_period_tables, ignore_index=True)

display(periodogram_summary)
display(fixed_period_summary)

## Plot a time-window and its periodogram

The left panels show the first three weeks of retained hourly bins.
The right panels show the periodogram over the full filtered window.

In [ ]:
plot_window_hours = 24 * 21
window = pc_scores[pc_scores["hours_since_start"] <= plot_window_hours]

fig, axes = plt.subplots(3, 2, figsize=(15, 10.5))
for row, component in enumerate(["PC1", "PC2", "PC3"]):
    ax = axes[row, 0]
    ax.plot(window["hours_since_start"], window[component], color="#2a9d8f", linewidth=1.1)
    ax.axhline(0, color="#8d99ae", linestyle="--", linewidth=1.0)
    ax.set_title(f"{component}: first {plot_window_hours // 24} days")
    ax.set_ylabel("Score")
    style_axes(ax)

    ax = axes[row, 1]
    result = periodogram_curves[component]
    ax.plot(period, result["power"], color="#1d3557", linewidth=2.0)
    ax.axvline(24.0, color="#e76f51", linestyle="--", linewidth=1.3, label="24 h")
    ax.axvline(result["best_period"], color="#264653", linestyle=":", linewidth=1.3, label=f"Peak {result['best_period']:.2f} h")
    ax.set_title(f"{component}: Lomb-Scargle periodogram")
    ax.set_xlabel("Period (hours)")
    ax.set_ylabel("Power")
    ax.legend(frameon=False)
    style_axes(ax)

axes[-1, 0].set_xlabel("Hours since first retained bin")
fig.tight_layout()
plt.show()
conn.close()